# TP : Streaming avec Kafka

Installation du client Kafka et imports :

In [ ]:
%pip install kafka-python==2.3.0

In [ ]:
from kafka import KafkaConsumer
import json

Configuration de la connexion :

In [ ]:
KAFKA_SERVER = '12.34.56.78:9092'

## Etape 1

### Prise en main

Dans cette première étape, on se contente de recevoir et d'afficher des messages qui contiennent un diamètre et un poids de pomme.

Choisir un nom de topic unique pour ce canal :

In [ ]:
TOPIC_DIAMETERS_AND_WEIGHTS = 'un-nom-orginal'

Reportez-le dans le script fourni `producer_1.py`.

Ensuite, lancez le script `producer_1.py` (par exemple dans un terminal de Jupyter).

Puis exécutez la cellule ci-dessous pour vérifier que les événements s'affichent bien !

**SI VOUS NE VOYEZ RIEN VENIR** : vérifiez d'abord les paramètres : adresse du serveur, et nom du topic. Ils doivent être identiques entre le notebook et les scripts de production. S'ils sont corrects, essayez d'interrompre la cellule et de la relancer juste après.

Remarquez que par défaut, la boucle ne rattrape pas les messages générés par les scripts avant qu'on ne l'exécute. Ce n'est pas gênant ici. Si besoin, on pourrait configurer le "consumer" pour commencer à lire dans le passé.

In [ ]:
consumer = KafkaConsumer(TOPIC_DIAMETERS_AND_WEIGHTS, bootstrap_servers=KAFKA_SERVER)

for message in consumer:
    # Le message est de type `bytes`, il faut le décoder en chaîne avant de le lire en JSON
    message_data = json.loads(message.value.decode())

    apple_id = message_data['apple_id']
    diameter = message_data['diameter']
    weight = message_data['weight']

    print(f'Apple {apple_id} => diameter = {diameter} cm, weight = {weight} g')

Vous pouvez interrompre la cellule et le script de production pour passer à la suite.

### Aiguillage

Ecrire une fonction qui décidera où aiguiller une pomme en fonction de son diamètre et de son poids. La fonction doit prendre ces 2 informations en argument, et renvoyer une chaîne de caractères qui dépend de la valeur des paramètres.

- Exemple 1 : si poids / diamètre > 11 alors `Chaine1` else `Chaine2`
- Exemple 2 : si poids < 100 et diamètre > 7 alors... sinon si ...

(la règle n'a pas besoin d'être réaliste !)

In [ ]:
def rule(diameter: float, weight: float) -> str:
    ...

Reprendre la boucle d'affichage ci-dessus pour émettre la décision en plus des infos reçues. Comme précédemment, exécutez d'abord le script, puis la boucle.

In [ ]:
consumer = KafkaConsumer(TOPIC_DIAMETERS_AND_WEIGHTS, bootstrap_servers=KAFKA_SERVER)

for message in consumer:
    # Le message est de type `bytes`, il faut le décoder en chaîne avant de le lire en JSON
    message_data = json.loads(message.value.decode())

    ...

## Etape 2

### Séparation des diamètres et des poids

En situation réelle, les infos sont envoyées par 2 systèmes différents qui ne se parlent pas (caméra et balance), et c'est à nous de rapprocher les diamètres et les poids correspondant à une pomme donnée. On suppose que les 2 systèmes vont envoyer les infos dans le même topic qu'on a utilisé jusque-là.

Reportez le nom de votre topic dans les 2 scripts `producer_2_diameters.py` et `producer_2_weights.py`.

Ensuite, adaptez la boucle précédente du notebook pour tenir compte de la séparation des envois.

Pour tester, lancez `producer_2_diameters.py` et `producer_2_weights.py` dans 2 fenêtres de terminaux différentes, puis la cellule.

Voici une suggestion :
- initialiser un dictionnaire vide, qui contiendra le mapping `apple_id` -> `{"diameter": diameter, "weight": weight}`
- dans la boucle :
  - à la réception d'un message, créer ou enrichir le mapping de la pomme concernée, en ajoutant l'info portée par le message (diamètre ou poids)
  - si le mapping de la pomme est complet (i.e. si on a le diamètre et le poids) :
    - appliquer la règle d'aiguillage
    - afficher les infos de la pomme et le résultat de la décision
    - supprimer le mapping du dictionnaire (pour libérer la mémoire)
  - autrement, il sera enrichi plus tard quand le message complémentaire sera traité !

In [ ]:
...